# <font color='056938'> **Load modules** </font>

We will load into this notebook some modules with functions that will help you in the development of heuristics and metaheuristics for the solution of travelling salesman problemas

The module `random_generators.py` include the functions to deal with generating random numbers. The ultimate function of this module is to generate a random permutation (solution) of the tsp problem
* Random tour generation: `rand_permutation`

The module `tsp_tools` provides different functions to help from the data reading to the solution ploting
* Data reading: `read_data(filepath)`
> `filepath`: the route to the file with the instance's data
* Distance calculation: `calculate_distances`
* Data display: `print_TSPdata()
* Solution evaluation: `tsp_distance`
* Tour display: `plot_tsp_route()`
* Plot the search trayectory `plot_trace(trace)`

In [1]:
#!gdown 1Z81jdqyAq3kX9xbZWGgOwV2UHpGDbUbq
#!gdown 1xnT-vDHdSbXVAp-exyAFR8rxVSD_moIa


import random_generators as ran_gen
import tsp_tools as tools

# <font color='056938'> **Instance generation** </font>

## <font color='8EC044'> **Data reading** </font>

We will use the clasical test instances stored in  [TSPlib](http://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/), the format of these instances include a heading with the following information:

>NAME: berlin52

>TYPE: TSP

>COMMENT: 52 locations in Berlin (Groetschel)

>DIMENSION: 52

>EDGE_WEIGHT_TYPE: EUC_2D

>NODE_COORD_SECTION

Then, n lines with the coordinates of the cities in the following format (cityID, Coordinate X, Coordinate Y):

**id cX cY**

Knowing this structure we are going to read three files

* `prueba10.tsp`
* `berlin52.tsp`
* `a280.tsp`




First we load the datafile to the colab environment

Then we read the data file

In [2]:
data = tools.read_data("prueba10.tsp")             #Read the data
data

[[565.0, 575.0],
 [25.0, 185.0],
 [345.0, 750.0],
 [945.0, 685.0],
 [845.0, 655.0],
 [880.0, 660.0],
 [25.0, 230.0],
 [525.0, 1000.0],
 [580.0, 1175.0],
 [650.0, 1130.0]]

We get the size of the instance. That is, the number of nodes `n`

In [3]:
n = len(data)
n

10

## <font color='8EC044'> **Distance calculation** </font>

We are going to use the $\tt{scipy.spatial}$ package to calculate the distances trhough the function `calculate_distances()`



In [4]:
distances=tools.calculate_distances(data)
distances

array([[   0.        ,  666.10809934,  281.11385594,  395.6008089 ,
         291.20439557,  326.26676202,  640.8002809 ,  426.87820277,
         600.18747071,  561.47128155],
       [ 666.10809934,    0.        ,  649.32657423, 1047.09120902,
         945.14549145,  978.08486339,   45.        ,  956.15113868,
        1134.95594628, 1132.98278892],
       [ 281.11385594,  649.32657423,    0.        ,  603.51056329,
         508.9449872 ,  542.51728083,  610.5735009 ,  308.05843601,
         485.64390246,  487.26276279],
       [ 395.6008089 , 1047.09120902,  603.51056329,    0.        ,
         104.40306509,   69.64194139, 1026.36494484,  525.        ,
         611.00327331,  533.90073984],
       [ 291.20439557,  945.14549145,  508.9449872 ,  104.40306509,
           0.        ,   35.35533906,  923.59352531,  470.55817919,
         583.6308765 ,  513.4685969 ],
       [ 326.26676202,  978.08486339,  542.51728083,   69.64194139,
          35.35533906,    0.        ,  957.0397066 ,  491

## <font color='056938'> **1.1 Initial solution generation** </font>

As usual, we will use the Traveling Salesman Problem (TSP) as a toy problem for illustrate the functionality of these methods. The code in the modules we load makes use of the code from [Taillard](http://mistic.heig-vd.ch/taillard/articles.dir/taillard.html) (2023), which is, in turn, built upon [L'Ecuyer's](https://www.iro.umontreal.ca/~lecuyer/) random number generator for randomness.

It is used to generate a random solution. That is an inital random permutation:

In [5]:
sol=ran_gen.rand_permutation(n)
sol

[4, 2, 9, 1, 7, 5, 0, 6, 8, 3]

##<font color='056938'> **1.1.1. Solution evaluation** </font>

### <font color='8EC044'> **Objective function calculation** </font>

Now we define use function `tsp_distance()` to evaluate the objective function of a given permutation

For the case of the solution we previuosly generated it will be:

In [7]:
obj_fn = tools.tsp_distance(distances,sol)
print("Distance of the permutation --> " + str(obj_fn))

Distance of the permutation --> 6355.292985706332


## <font color='056938'> **1.2 Simulated annealing** </font>



El Simulated annealing es junto con las estrategias evolutivas el método precursor de las metaheurísticas. Dado que son muy similares, vamos a convertir una Estrategia Evolutiva (1+1) en un recocido simulado.

**Pseudocodigo de un recocido simulado**


![](https://drive.google.com/uc?id=12gnH4SDwqyNrkOMty53S0CGaofkUPkt5)

### <font color='8EC044'> **1.2.1 Como explorar el vecindario de las soluciones de un TSP?** </font>



La siguientes dos funciones modifican aleatoriamiente un tour de TSP  de dos maneras complementarias:

>Swap: Intercambiando dos ciudades al azar

>Insertion: Reinsertando una ciudad elegida al azar en otra posición tambien elegida alazar

### <font color=blue> **Swap**</font>

In [8]:
import random

def swap(tour):
  """
  Swaps the positions of two randomly selected cities in a TSP tour.

  Args:
    tour: A list representing a TSP tour (permutation of city indices).

  Returns:
    A new list representing the modified tour with two cities swapped.
  """
  # Create a copy to avoid modifying the original tour
  new_tour = tour[:]

  # Select two different random indices
  idx1, idx2 = random.sample(range(len(new_tour)), 2)

  # Swap the elements at the selected indices
  new_tour[idx1], new_tour[idx2] = new_tour[idx2], new_tour[idx1]

  return new_tour

Probemos la funcion swap

In [9]:
sol=ran_gen.rand_permutation(n)    #Generar una solución aleatoria
print("Solucion: " +  str(sol))
obj_fn = tools.tsp_distance(distances,sol)
print("Distancia de la solucion --> " + str(obj_fn))
swapSol=swap(sol)
print("Solucion mutada con swap: " +  str(swapSol))
obj_fn = tools.tsp_distance(distances,swapSol)
print("Distancia de la solucion mutada --> " + str(obj_fn))


Solucion: [7, 2, 3, 1, 0, 4, 5, 9, 8, 6]
Distancia de la solucion --> 5571.823768267195
Solucion mutada con swap: [7, 2, 0, 1, 3, 4, 5, 9, 8, 6]
Distancia de la solucion mutada --> 5062.625730434657


### <font color=blue> **Insert**</font>

In [10]:
import random

def insert(tour):
  """
  Selects a random position in a TSP tour and reinserts it into another random position.

  Args:
    tour: A list representing a TSP tour (permutation of city indices).

  Returns:
    A new list representing the modified tour with the element moved.
  """
  # Create a copy to avoid modifying the original tour
  new_tour = tour[:]

  # Select a random index to remove
  removed_idx = random.randint(0, len(new_tour) - 1)

  # Remove the element at the selected index
  removed_element = new_tour.pop(removed_idx)

  # Select a new random index to insert
  inserted_idx = random.randint(0, len(new_tour))

  # Insert the removed element at the new index
  new_tour.insert(inserted_idx, removed_element)

  return new_tour


Ahora probemos la funcion Insert

In [11]:
sol=ran_gen.rand_permutation(n)    #Generar una solución aleatoria
print("Solucion: " +  str(sol))
obj_fn = tools.tsp_distance(distances,sol)
print("Distancia de la solucion --> " + str(obj_fn))
insertSol=insert(sol)
print("Solucion mutada con insert: " +  str(insertSol))
obj_fn = tools.tsp_distance(distances,insertSol)
print("Distancia de la solucion mutada --> " + str(obj_fn))

Solucion: [3, 6, 5, 4, 7, 2, 9, 8, 1, 0]
Distancia de la solucion --> 5564.520807886158
Solucion mutada con insert: [3, 6, 5, 4, 7, 2, 9, 8, 1, 0]
Distancia de la solucion mutada --> 5564.520807886158


## <font color=blue> Evolutionary strategy (ES) <font>

Let's define the ES function:

In [12]:
import random
import tsp_tools as tools

def evolutionary_strategy(initial_solution, distances, iterations):
  """
  Applies the insert operator for a given number of iterations, updating the
  solution if an improvement is found. This implements a (1+1) Evolution Strategy.

  Args:
      initial_solution: The initial TSP tour (a list of city indices).
      distances: The pre-calculated distance matrix between cities.
      iterations: The number of iterations to run the strategy for.

  Returns:
      A tuple containing:
          - The best solution found during the process (a list of city indices).
          - The distance of the best solution found.
          - A list of distances of the solution at each improvement.
  """
  current_solution = initial_solution[:]  # Create a copy to avoid modifying the original
  best_solution = initial_solution[:]
  current_distance = tools.tsp_distance(distances, current_solution)
  best_distance = current_distance
  trace = []

  for _ in range(iterations):
    # Apply the insertion mutation
    mutated_solution = insert(current_solution)

    # Evaluate the mutated solution
    mutated_distance = tools.tsp_distance(distances, mutated_solution)

    # Check if the mutated solution is better
    if mutated_distance < current_distance:
      current_solution = mutated_solution
      current_distance = mutated_distance
      trace.append(current_distance)

      #update best solution if needed
      if current_distance < best_distance:
        best_solution = current_solution[:]
        best_distance = current_distance

  return best_solution, best_distance, trace



In [13]:
# Example usage (assuming you have data and distances calculated):
# Assuming you have already calculated the following variables:
data = tools.read_data("prueba10.tsp")
distances=tools.calculate_distances(data)
n = len(data)

sol=ran_gen.rand_permutation(n)
print("Solucion Inicial: " +  str(sol))
obj_fn = tools.tsp_distance(distances,sol)
print("Distancia de la solucion inicial--> " + str(obj_fn))

best_solution, best_distance, trace = evolutionary_strategy(sol, distances, 100)
print("Mejor solucion:", best_solution)
print("Mejor distance:", best_distance)
print("Trace:", trace)

Solucion Inicial: [2, 9, 1, 0, 4, 5, 8, 6, 3, 7]
Distancia de la solucion inicial--> 6164.268584279823
Mejor solucion: [5, 3, 8, 9, 7, 2, 6, 1, 0, 4]
Mejor distance: 2900.5084584334763
Trace: [5938.235949276996, 5899.1525953552045, 5505.435147594813, 4933.069153169767, 3798.744544397872, 3798.7445443978713, 3458.7938065456065, 3458.793806545606, 3400.3878022761915, 3295.975619679222, 3199.6820909099256, 2900.5084584334763]


## <font color=red> Simulated annealing (SA) <font>


*   Usa el código de la estrategia evolutiva de arriba y convuertelo en un recocido simulado
*   Compara los resultados en alguna de las instancias de TSP que tenemos a la mano



